# 04 Pdf Processing

Companion notebook to the textbook. Run from the `notebooks/` folder inside the project (the code uses `..` paths). Cells marked NEEDS INTERNET download a Hugging Face model on first run.

# Chapter 11: PDF Processing

## 11.1 PDF structure in simple terms

## 11.2 Extracting text and metadata page by page

In [ ]:
import sys
sys.path.insert(0, "..")
from pypdf import PdfReader
from src.pdf_processor import extract_pages, extract_metadata
path = "../data/sample_pdfs/ml_basics.pdf"
print(extract_metadata(path))
reader = PdfReader(path)
raw = reader.pages[3].extract_text()
print(repr(raw[:120]))
pages = extract_pages(path)
print([(p.page_number, len(p.text)) for p in pages])

## 11.3 Empty pages, bad files and validation

In [ ]:
import sys, tempfile, os
sys.path.insert(0, "..")
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from src.pdf_processor import extract_pages, validate_pdf, PDFValidationError
d = tempfile.mkdtemp()
p = os.path.join(d, "with_blank.pdf")
c = canvas.Canvas(p, pagesize=A4)
c.drawString(72, 750, "Page one has text."); c.showPage()
c.showPage()                                    # page two is blank
c.drawString(72, 750, "Page three has text."); c.showPage()
c.save()
print([(x.page_number, x.text) for x in extract_pages(p)])
fake = os.path.join(d, "fake.pdf"); open(fake, "wb").write(b"hello, not a pdf")
try:
    validate_pdf(fake)
except PDFValidationError as e:
    print("rejected:", e)

## 11.4 Limitations and how to handle them

## 11.5 OCR (optional extension, not implemented in this project)

## 11.6 Interview questions and answers

# Chapter 12: Text Chunking

## 12.1 What chunking is and why we need it

## 12.2 Strategies in this project (`src/chunking.py`)

## 12.3 Chunking a real page with each strategy

In [ ]:
import sys
sys.path.insert(0, "..")
from src.pdf_processor import extract_pages
from src.chunking import fixed_size_split, sentence_split, recursive_split
text = extract_pages("../data/sample_pdfs/ml_basics.pdf")[3].text      # gradient descent page
print("page length:", len(text), "characters")
for name, fn in (("fixed", fixed_size_split), ("sentence", sentence_split), ("recursive", recursive_split)):
    chunks = fn(text, 300, 50)
    print(f"{name:9s} -> {len(chunks)} chunks, lengths {[len(c) for c in chunks]}")
print("--- first fixed chunk ends:", repr(fixed_size_split(text, 300, 50)[0][-40:]))
print("--- first sentence chunk ends:", repr(sentence_split(text, 300, 50)[0][-40:]))

## 12.4 Comparing chunking strategies on retrieval (measured)

In [ ]:
import sys
sys.path.insert(0, "..")
from evaluation.evaluate_retrieval import evaluate
for strategy in ("fixed", "sentence", "recursive"):
    for size in (200, 800):
        r = evaluate("hashing", strategy, size, size // 6, 4)
        print(f"{strategy:9s} size={size:3d} hit@4={r['hit@4']}  precision@4={r['precision@4']}  MRR={r['MRR']}")

## 12.5 Practical guidance

## 12.6 Interview questions and answers